## Step 0 — Gradient Checker

In [24]:
from __future__ import annotations  # lazy annotations: allows tuple[...] / X | None on older kernels

from typing import Callable
import numpy as np

In [25]:
def numeric_gradient(
    scalarFunction: Callable[[np.ndarray], float],
    featureValues: np.ndarray,
    h: float = 1e-5
) -> np.ndarray:
    gradient = np.zeros_like(featureValues, dtype=float)

    for i in range(featureValues.size):
        mask = np.zeros_like(featureValues, dtype=float)
        mask.flat[i] = h
        gradient.flat[i] = (scalarFunction(featureValues + mask) - scalarFunction(featureValues - mask)) / (2 * h)
    return gradient

def stable_softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

### Tests — `numeric_gradient`

In [26]:
def _check(name, got, want, atol=1e-6):
    ok = np.allclose(got, want, atol=atol)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", got)
        print("   want:", want)

# 1. f = sum(x^2)  ->  grad = 2x        (vector input, nonlinear)
x = np.array([3.0, -1.0, 0.5])
_check("sum(x^2) grad == 2x", numeric_gradient(lambda v: np.sum(v**2), x), 2 * x)

# 2. f = sum(c*x)  ->  grad = c         (linear -> constant gradient)
c = np.array([2.0, -3.0, 0.7])
_check("sum(c*x) grad == c", numeric_gradient(lambda v: np.sum(c * v), np.zeros(3)), c)

# 3. matrix input -> gradient keeps the matrix shape
W = np.arange(6, dtype=float).reshape(2, 3)
g = numeric_gradient(lambda M: np.sum(M**2), W)
_check("matrix grad == 2W", g, 2 * W)
_check("matrix grad keeps shape", np.array(g.shape), np.array(W.shape))

# 4. f = sum(sin x) -> grad = cos x     (check vs analytic nonlinear)
x = np.array([0.1, 0.7, -1.2, 2.0])
_check("sum(sin x) grad == cos x", numeric_gradient(lambda v: np.sum(np.sin(v)), x), np.cos(x))

[PASS] sum(x^2) grad == 2x
[PASS] sum(c*x) grad == c
[PASS] matrix grad == 2W
[PASS] matrix grad keeps shape
[PASS] sum(sin x) grad == cos x


### Tests — `stable_softmax`

In [27]:
# reuses _check from the numeric_gradient test cell above
x = np.array([2.0, 1.0, 0.1])
p = stable_softmax(x)

_check("probs sum to 1", p.sum(), 1.0)
_check("all in (0, 1)", np.all((p > 0) & (p < 1)), True)

# matches the naive definition on small, safe inputs
naive = np.exp(x) / np.sum(np.exp(x))
_check("matches naive softmax", p, naive)

# stability + shift-invariance: huge logits stay finite and give the same result
big = stable_softmax(x + 1000)
_check("finite on x + 1000", np.all(np.isfinite(big)), True)
_check("shift-invariant (== p)", big, p)

# monotonic: largest logit keeps the largest probability
_check("argmax preserved", np.argmax(p), np.argmax(x))

[PASS] probs sum to 1
[PASS] all in (0, 1)
[PASS] matches naive softmax
[PASS] finite on x + 1000
[PASS] shift-invariant (== p)
[PASS] argmax preserved


## Step 1 — Linear Layer

In [28]:
class Linear:
    """Fully-connected layer:  Y = X @ W + b

    Shapes:
        X : (batch, n_in)     activations from the previous layer
        W : (n_in, n_out)
        b : (n_out,)
        Y : (batch, n_out)
    """

    def __init__(self, n_in: int, n_out: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        # simple init: standard normal weights, zero bias
        self.W: np.ndarray = rng.standard_normal((n_in, n_out))
        self.b: np.ndarray = np.zeros(n_out)
        # caches / gradient buffers (filled during forward/backward)
        self.X: np.ndarray | None = None   # previous layer's activations, saved for backward
        self.dW: np.ndarray | None = None
        self.db: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (batch, n_in) -> Y: (batch, n_out)."""
        self.X = X                     # cache X: backward needs it to compute dW
        return X @ self.W + self.b     # b (n_out,) broadcasts across every row

    def backward(self, dY: np.ndarray) -> np.ndarray:
        """dY: (batch, n_out) upstream gradient dL/dY. Returns dX: (batch, n_in)."""
        # dW: chain rule + sum over the batch -> X.T @ dY.
        #   local derivative dY/dW is X; a weight is reused across all samples,
        #   so the matmul sums those per-sample contributions. Shape (n_in, n_out).
        self.dW = self.X.T @ dY

        # db: local derivative dY/db is 1, so just sum dY over the batch axis.
        #   Shape (n_out,) -- one gradient per bias.
        self.db = dY.sum(axis=0)

        # dX: gradient to hand back to the previous layer (becomes its dY).
        #   local derivative dY/dX is W, so dX = dY @ W.T. Shape (batch, n_in).
        dX = dY @ self.W.T
        return dX

### Test — `Linear` gradient check

In [29]:
# Uses numeric_gradient + _check from Step 0.
# Trick: wrap the layer in a SCALAR loss  L = sum(Y * dY)  (so dL/dY = dY),
# then numeric_gradient of L w.r.t. each of W, b, X must match backward().
def gradient_check_linear(n_in=4, n_out=3, batch=5, seed=1):
    rng = np.random.default_rng(seed)
    layer = Linear(n_in, n_out)
    X  = rng.standard_normal((batch, n_in))
    dY = rng.standard_normal((batch, n_out))     # random upstream (not all-ones)

    W0, b0 = layer.W.copy(), layer.b.copy()

    # analytic gradients from the layer's own backward
    layer.forward(X)
    dX = layer.backward(dY)
    dW_a, db_a = layer.dW.copy(), layer.db.copy()

    # numeric dW: vary W, hold X/b fixed
    def loss_W(Wf):
        layer.W = Wf.reshape(W0.shape)
        return np.sum(layer.forward(X) * dY)
    dW_n = numeric_gradient(loss_W, W0.copy()); layer.W = W0.copy()

    # numeric db: vary b
    def loss_b(bf):
        layer.b = bf.reshape(b0.shape)
        return np.sum(layer.forward(X) * dY)
    db_n = numeric_gradient(loss_b, b0.copy()); layer.b = b0.copy()

    # numeric dX: vary X, params at originals
    def loss_X(Xf):
        return np.sum(layer.forward(Xf.reshape(X.shape)) * dY)
    dX_n = numeric_gradient(loss_X, X.copy())

    _check("dLinear/dW", dW_a, dW_n)
    _check("dLinear/db", db_a, db_n)
    _check("dLinear/dX", dX,   dX_n)

gradient_check_linear()

[PASS] dLinear/dW
[PASS] dLinear/db
[PASS] dLinear/dX


## Step 2 — Cross-Entropy Loss

In [30]:
def cross_entropy(
    rawClassScores: np.ndarray,
    correctClassIndices: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Softmax cross-entropy for a batch of classification examples.

    Args:
        rawClassScores      : (numberOfExamplesInBatch, numberOfClasses)
                              raw scores (logits), one row per example
        correctClassIndices : (numberOfExamplesInBatch,)
                              correct class index for each example

    Returns:
        meanLoss          : float        mean cross-entropy over the batch
        gradientWrtScores : np.ndarray   (numberOfExamplesInBatch, numberOfClasses)
                            dL/dScores = (softmax - onehot) / numberOfExamplesInBatch
    """
    numberOfExamplesInBatch: int = rawClassScores.shape[0]
    exampleRows: np.ndarray = np.arange(numberOfExamplesInBatch)   # [0, 1, ..., batch-1], to index each row

    # --- stable softmax, per row (subtract each row's max -> no overflow) ---
    stabilizedScores: np.ndarray = rawClassScores - rawClassScores.max(axis=1, keepdims=True)
    exponentiatedScores: np.ndarray = np.exp(stabilizedScores)
    classProbabilities: np.ndarray = exponentiatedScores / exponentiatedScores.sum(axis=1, keepdims=True)

    # --- loss: -log(prob of the correct class), averaged over the batch ---
    probabilityOfCorrectClass: np.ndarray = classProbabilities[exampleRows, correctClassIndices]
    negativeLogProbOfCorrectClass: np.ndarray = -np.log(probabilityOfCorrectClass)   # (batch,)
    meanLoss: float = float(negativeLogProbOfCorrectClass.mean())

    # --- gradient: the clean combined form  softmax - onehot  ---
    #   start from probabilities, subtract 1 at each row's correct class, average over batch
    gradientWrtScores: np.ndarray = classProbabilities.copy()
    gradientWrtScores[exampleRows, correctClassIndices] -= 1
    gradientWrtScores /= numberOfExamplesInBatch
    return meanLoss, gradientWrtScores

### Test — `cross_entropy` gradient check

In [31]:
# The combined gradient softmax - onehot must match finite differences of the loss.
def gradient_check_cross_entropy(numberOfExamplesInBatch=4, numberOfClasses=5, seed=1):
    randomGenerator = np.random.default_rng(seed)
    randomScores = randomGenerator.standard_normal((numberOfExamplesInBatch, numberOfClasses))
    correctClassIndices = randomGenerator.integers(0, numberOfClasses, size=numberOfExamplesInBatch)

    analyticLoss, analyticGradient = cross_entropy(randomScores, correctClassIndices)

    # numeric gradient: loss as a scalar function of the scores
    def lossAsFunctionOfScores(flattenedScores):
        lossValue, _ = cross_entropy(flattenedScores.reshape(randomScores.shape), correctClassIndices)
        return lossValue
    numericGradient = numeric_gradient(lossAsFunctionOfScores, randomScores.copy())

    _check("dCE/dScores", analyticGradient, numericGradient)

gradient_check_cross_entropy()

[PASS] dCE/dScores
